# Week 9 — Monday: Preparing the Data — Scaling, Encoding and Pipelines

**DATA 202 · Calvin University**

April 1912: the RMS *Titanic* sank on its first voyage. Its passenger list survives: 1,309 passengers, with their ticket class, sex, age, family aboard, fare and port.

> 🎯 **Can kNN learn who survived from a list that mixes years, pounds and words?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · The passenger list |
| ~10 min | 2 · Years vs. pounds: scaling (SLO 09A) |
| ~12 min | 3 · Words to numbers: one-hot encoding (SLO 09A) |
| ~10 min | 4 · One pipe for every step: pipelines (SLO 09B) |
| ~3 min | 5 · Who did we leave out? |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: *titanic3*, compiled by Thomas Cason from Encyclopedia Titanica; distributed by Frank Harrell, Vanderbilt Biostatistics</sub>

---
## 1 · The Passenger List · ~5 min

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

titanic = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/titanic.csv")
titanic.head()

| column | means |
|---|---|
| `pclass` | ticket class: 1st, 2nd or 3rd |
| `sex` | female or male |
| `age` | in years |
| `sibsp` | siblings + spouses aboard |
| `parch` | parents + children aboard |
| `fare` | ticket price, in 1912 pounds (£) |
| `cabin` | cabin number |
| `embarked` | port: Southampton, Cherbourg or Queenstown |
| `survived` | **the label**: 1 = survived, 0 = died |

In [ ]:
passengers = titanic.dropna(subset=["age", "fare", "embarked"])     # today: passengers with these recorded
X = passengers[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
y = passengers["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)
print(len(passengers), "passengers |", len(X_train), "training |", len(X_test), "test")

Today: the 1,043 passengers whose age, fare and port were recorded. **Wednesday:** the 266 we just left out.

> 💬 **Question:** kNN measures distances between rows of numbers. Which columns can it use as they are?

<details><summary>Answer</summary>

* **Numbers:** `age`, `sibsp`, `parch`, `fare`, and `pclass`
* **Words:** `sex` and `embarked`. What is "female" minus "male"?
* Even the numbers aren't ready: years, people and pounds are different rulers (Part 2). And is `pclass` really a number (Part 3)?
</details>

---
## 2 · Years vs. Pounds: Scaling (SLO 09A) · ~10 min

In [ ]:
numeric = ["pclass", "age", "sibsp", "parch", "fare"]
fig = px.box(X_train[numeric], title="Five Numeric Columns, Five Different Rulers (782 training passengers)")
fig.show()

Fares run from £0 to £512. Ages stop at 80; family counts at 8.

> 📄 **Handout 1 — Who is closest?** Three training passengers. Is A closer to B or to C? First in years and pounds, then in standard deviations. 🗣️ The plots disagree: which answer would you trust?

In [ ]:
names = ["Colley, Mr. Edward Pomeroy", "Davies, Mr. John Samuel", "Chaffee, Mr. Herbert Fuller"]
three = passengers.set_index("name").loc[names, ["pclass", "age", "fare"]]
three.index = ["A", "B", "C"]
display(three)

def distance(p, q):
    return ((p["age"] - q["age"]) ** 2 + (p["fare"] - q["fare"]) ** 2) ** 0.5

raw = three[["age", "fare"]]
in_sd = raw / X_train[["age", "fare"]].std()       # each column divided by its standard deviation

print("Years and pounds:       A to B =", round(distance(raw.loc["A"], raw.loc["B"]), 1),
      "| A to C =", round(distance(raw.loc["A"], raw.loc["C"]), 1))
print("Standard deviations:    A to B =", round(distance(in_sd.loc["A"], in_sd.loc["B"]), 2),
      "| A to C =", round(distance(in_sd.loc["A"], in_sd.loc["C"]), 2))

<details><summary>Answer</summary>

* **Years and pounds:** B is closer (26.0 vs. 35.6). B is 26 years younger, but his fare is almost the same, and £1 counts as much as 1 year
* **Standard deviations** (age ÷ 14.5, fare ÷ 50.3): C is closer (0.71 vs. 1.79). C is A's age, and £36 is less than one standard deviation of fare
* Same three people, a different nearest neighbor. The **units** chose it
* Trust the scaled answer: C shares A's class and age; B is 26 years younger and in 3rd class
</details>

**`StandardScaler`**, one column at a time: z = (value − mean) ÷ standard deviation

* Every column ends up with mean 0 and standard deviation 1: one ruler for all
* The mean and standard deviation come from the **training** passengers only

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

scaler = StandardScaler().fit(X_train[numeric])      # learns each column's mean and sd (training passengers)
train_scaled = scaler.transform(X_train[numeric])
test_scaled = scaler.transform(X_test[numeric])      # the test passengers get the SAME mean and sd

def test_accuracy(model, train, test):
    model.fit(train, y_train)
    return round(model.score(test, y_test), 3)

knn = KNeighborsClassifier(n_neighbors=5)
tree = DecisionTreeClassifier(max_depth=4, random_state=0)
print("kNN,  raw:   ", test_accuracy(knn, X_train[numeric], X_test[numeric]))
print("kNN,  scaled:", test_accuracy(knn, train_scaled, test_scaled))
print("tree, raw:   ", test_accuracy(tree, X_train[numeric], X_test[numeric]))
print("tree, scaled:", test_accuracy(tree, train_scaled, test_scaled))

<details><summary>Answer</summary>

* **kNN:** 66.3% → 68.6% after scaling. It measures distances, so units matter
* **Tree:** 71.6% both ways. Each question looks at one column ("fare ≤ 52?"), so the units never meet
* Scale for models that measure distance (kNN, k-means, PCA)
</details>

> 📌 **One split:** test accuracy on *these* 261 test passengers.

---
## 3 · Words to Numbers: One-Hot Encoding (SLO 09A) · ~12 min

kNN needs numbers, but `embarked` holds words: Southampton, Cherbourg, Queenstown.

> 🎯 **Predict First:** Quick fix: number the ports, Southampton = 1, Cherbourg = 2, Queenstown = 3. What would kNN now believe about the three ports?

<details><summary>Answer</summary>

* Queenstown is **2** away from Southampton, but Cherbourg only **1**: Cherbourg sits "between" them
* The ports have no such order. We invented it when we picked the numbers
* Number them in another order and kNN finds different neighbors
</details>

**One-hot encoding:** one new 0/1 column per category. Each row gets a 1 in its own category's column and 0 in the rest: exactly one column is "hot".

| embarked | → | embarked_Cherbourg | embarked_Queenstown | embarked_Southampton |
|---|---|:-:|:-:|:-:|
| Southampton | | 0 | 0 | **1** |
| Cherbourg | | **1** | 0 | 0 |
| Queenstown | | 0 | **1** | 0 |

* The columns are also called **dummy variables** (`pd.get_dummies` in pandas)
* One word column becomes as many columns as it has categories

> 📄 **Handout 2 — One-hot encode four passengers.** Fill in the 0s and 1s for `sex` and `embarked`. 🗣️ Then talk with your neighbor: one-hot makes every port equally far from every other. Is that fair to the three ports?

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="if_binary", sparse_output=False).fit(X_train[["sex", "embarked"]])
four = X_train[["sex", "embarked"]].drop_duplicates().head(4)          # four different combinations
display(four)
pd.DataFrame(encoder.transform(four), columns=encoder.get_feature_names_out(), index=four.index).astype(int)

<details><summary>Answer</summary>

* `embarked` → 3 columns, with exactly one 1 in each row
* `sex` → **one** column, `sex_male`: 1 = male, 0 = female. A `sex_female` column would just be 1 − `sex_male`, so `drop="if_binary"` leaves it out
* Distances on the port columns (two rows with different ports differ in two columns: √(1² + 1²)):

| ports | numbered 1, 2, 3 | one-hot |
|---|:-:|:-:|
| Southampton – Cherbourg | 1 | 1.41 |
| Cherbourg – Queenstown | 1 | 1.41 |
| Southampton – Queenstown | **2** | 1.41 |

* One-hot: every port is **equally far** from every other port. Same port → 0
</details>

<details><summary>Is that fair to the ports?</summary>

* The ports were not alike (training passengers): Cherbourg was 62% 1st class, Queenstown 80% 3rd class (median fare £7.79), Southampton a mix
* One-hot says nothing about that, and it doesn't need to: kNN learns it from the **other** columns (`pclass`, `fare`) and from who survived
* Encoding is the honest "we don't know how these differ"; numbering the ports would claim we do
</details>

**`OneHotEncoder`**, one column at a time:

* `fit` learns the list of categories from the **training** passengers
* `transform` writes one 0/1 column per category
* `drop="if_binary"`: a column with only two categories becomes one 0/1 column

`pclass` is already 1, 2, 3. Is it a **rank** or a **name**?

> "From the point of view of social order, the most basic kinds of processes are those of nominalization and ordinalization: the naming of kinds and the designation of ranks. … In practice, ranks can be collapsed into types."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Keep `pclass` as one ranked column (1 < 2 < 3), or one-hot it into three kinds? What does each choice tell kNN about the passengers?

<details><summary>Answer</summary>

* **Rank (one column):** 1st class sits closer to 2nd than to 3rd. That fits fares, decks and distance from the lifeboats
* **Kinds (three columns):** every class is equally far from every other
* Neither is "the data". It's a choice we make. Today we keep the ranks. For words with an order ("low", "medium", "high") there is `OrdinalEncoder`
</details>

---
## 4 · One Pipe for Every Step: Pipelines (SLO 09B) · ~10 min

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/pipeline.png" width="380" alt="A drawing of connected pipes: water goes in at one end, through many bends and joints, and out at the other.">

A **pipeline** joins the steps: each step's output flows into the next one. A **column transformer** sends different columns down different pipes.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric),                          # ("name", step, columns)
    ("cat", OneHotEncoder(drop="if_binary"), ["sex", "embarked"]),
])
model = Pipeline([
    ("prep", preprocess),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
model                                                            # click the boxes to open them

> 📄 **Handout 3 — Follow the pipe.** Write which columns go into each branch, and how many columns come out of `prep` into kNN.

In [ ]:
model.fit(X_train, y_train)
print(model.named_steps["prep"].get_feature_names_out())
print("test accuracy:", round(model.score(X_test, y_test), 3))

<details><summary>Answer</summary>

* 5 scaled numbers + `sex_male` + 3 ports = **9 columns** into kNN
* 68.6% → **76.2%**. The most useful column was a word: "women and children first"
</details>

**Why a pipeline?**

1. **One object:** `fit` and `predict` run every step, in order. In Part 2 we had to scale the training *and* the test passengers by hand
2. **No leaks:** `fit` learns the scaler's means and the encoder's categories from the training rows only; the test rows just flow through. Practice 05's rule — test rows never go into `fit` — kept automatically
3. **Easy to change:** swap a step or a knob, and the rest stays the same

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, {"knn__n_neighbors": [1, 3, 5, 9, 15, 25, 51, 101, 201]},   # "step__knob"
                    cv=5, return_train_score=True)
grid.fit(X_train, y_train)               # training passengers only: 9 values of k × 5 folds

curve = pd.DataFrame({"k": grid.cv_results_["param_knn__n_neighbors"],
                      "training folds": grid.cv_results_["mean_train_score"],
                      "cross-validated": grid.cv_results_["mean_test_score"]})
fig = px.line(curve, x="k", y=["training folds", "cross-validated"], markers=True, log_x=True,
              labels={"value": "accuracy", "variable": "scored on"}, title="Choosing k for the Whole Pipeline")
fig.show()

> 📄 **Handout 4 — Choose k.** Circle the best k. Label the overfitting and underfitting sides.

<details><summary>Answer</summary>

* **Best: k = 25** (77.2% cross-validated); k = 15 is right behind (77.0%)
* **k = 1:** 98.8% on the training folds, 71.4% cross-validated → **overfitting**
* **k = 201:** about 70% on both → **underfitting**
* Every k, every fold refit the whole pipe: the scaler and encoder learned from 4 folds, and the 5th fold just flowed through. No leaks, even inside cross-validation
</details>

Only now do the test passengers come out:

In [ ]:
print("best k:", grid.best_params_["knn__n_neighbors"], "| cross-validated:", round(grid.best_score_, 3))
print("test accuracy:", round(grid.score(X_test, y_test), 3))      # refit on all training passengers, scored once
print("lazy model, 'everyone died':", round((y_test == 0).mean(), 3))

* **81.2%** on the test passengers, vs. **59.4%** for the lazy model
* We chose k without looking at the test passengers, so 81.2% is a fair score for *this* model

> 📌 **One split:** 81.2% is on *these* 261 test passengers. Cross-validation said 77.2%, with a spread of about ±3.5 points across folds: the test split landed on the lucky side.

---
## 5 · Who Did We Leave Out? · ~3 min

We started with 1,309 passengers and kept 1,043. The model never saw the other **266**.

> 🎯 **Predict First (for Wednesday):** Were the 266 like everyone else? If not, who were they?

---
### Wrap-up

1. **Scale** columns measured on different rulers before measuring distances; trees don't need it
2. **One-hot encode** categories with no order; keep or build ranks only when the order is real, and say so
3. A **pipeline** chains the steps and fits them on training rows only; new rows just flow through
4. **Grid search** tunes a knob of the whole pipeline, refitting every step inside every fold
5. 📌 Every score: one dataset, one split, and only the passengers who were fully recorded

**Wednesday:** missing data — the 266.